In [ ]:
!pip install ipympl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML
from functools import partial
from scipy.fft import fft, fftshift, fftfreq, rfft, rfftfreq

In [ ]:
"""Setting up the domain"""
L = 10 # length of the domain
xs = np.arange(-L/2,L/2, L/256) # x in [0,L), 256 steps

In [ ]:
L  = 10  # 2pi/delta_k
nx = 256
dx = L/nx  # 2pi/k_max...called the "Nyquist" wavenumber
xs = np.arange(-L/2, L/2, dx) # x=0 at the centre of the domain

In [ ]:
# slow Fourier transform
def slow_fourier(f,L, xs, m=10):
    dx = xs[1]-xs[0]
    delta_k = 2*np.pi/L
    k_max = 2*m*np.pi/L # 2mpi/L, where m < nx
    k = -k_max
    A = []
    B = []
    ks = []
    while k <= k_max:
        ks.append(k)
        A.append(np.sum(f*np.cos(k*xs))*dx/L)
        B.append(np.sum(f*np.sin(k*xs))*dx/L)
        k += delta_k
    A = np.array(A)
    B = np.array(B)    
    return (ks, A, B)

In [ ]:
# Fourier reconstruction
f16 = np.zeros_like(xs)
ks, A, B = slow_fourier(f, L, xs, 16)
for i, k in enumerate(ks):
    f16 += A[i]*np.cos(ks[i]*xs) + B[i]*np.cos(ks[i]*xs)
plt.plot(f16)


### Exercise 2

#### 2.1 Square wave

Construct a square wave given by the function $f(x)$ which equals $-1$ for $x\in(-1,0)$ and $1$ for $x\in (0,1).$ and assume $L=2.$
(a) Find the Fourier transform of this wave analytically.
(b) Find its Fourier transform $\hat f(k) = A+iB$ using the code for the slow Fourier transform. Vary $L$ and $\mathrm dx$ to get fewer or more wavenumbers. Then carry out a *Fourier synthesis*, reconstructing the original waveform as $$f(x)\approx\sum_kA_k\cos(kx)+B_k\sin(kx).$$ How many wavenumbers do you need to construct a good reconstruction? At what points does the reconstruction suck the most? Write a function (based off the slow fourier transform code above) to produce multiple such reconstructions for fewer or more wavenumbers, and compare plots.


#### 2.2 Gaussian wave

Visualise a Gaussian given by $$f(x)=e^{-ax^2}.$$ Verify that the absolute value of the complex-valued Fourier transform $|\hat f(k)|$ matches $$\frac{2\sqrt\pi}{aL}e^{-k^2/4a^2}$$ 

(a) Notice the sine coefficients $B(k)$ are identically 0; why? What happens if you interchange the sine and cosine coefficients while reconstructing the function: again, why? 

(b)  For a given $a,$ what is a reasonable choice of $L$ and $dx$ to carry out the same exercise as the square wave?How does this number change when the real space Gaussian is thinner or thicker?

(c) Similarly, if the physical width of the Gaussian is the same but the artificial length scale L over which periodicity is assumed changes, what are the consequences for the number of Fourier modes you need to sum to get a good representation?

#### 2.3 Wavepacket

Consider a new kind of waveform called a *wavepacket*: a wave multiplied by a Gaussian envelope $\sin(k_0x)e^{-ax^2}$. 

(a)Substitute the Fourier expansion for the Gaussian into the envelope factor, and then apply trigonometric identities to verify that the Fourier transform of such a function is a Gaussian in $k$-space, but centred at $k_0$ rather than $0.$ 

(b) Verify this using the code as well.


#### 2.4 FFT and RFFT

Code snippets for this exercise are given below for you to tinker with.

Play with the `fft` function for the waveforms discussed above, and verify that you get the same results as via the 
slow Fourier transform. Ensure that you are able to use `fftshift` correctly. If you take just the `rfft`, note that you get the transform for $k\geq 0$ only. Note you are never asked for `xs`, just `f` and `dx`: thus SciPy *assumes* a certain location of $x=0.$ Explore how this causes it to differ from our "slow" Fourier transform, by changing `xs` to go from 0 to $L$ or from $-L$ to 0, and plotting the reconstruction. Does this also happen if you reconstruct it using their inbuilt function `irfft` (or `ifft`)? Note that `ifft` requires the wavenumbers to be in their funky order, i.e., you may need to uneo the shift.

In [ ]:
# rfft example
f = # TODO provide a function of x
m =  # TODO number of fourier components
_f = rfft(f, m) # note it does NOT ask for xs
ks =  rfftfreq(m, d=dx)
A = np.real(_f)
B = np.imag(_f)

In [ ]:
# fft example
f = # TODO provide a function of x
m =  # TODO number of fourier components
_f = fft(f, m) # note it does NOT ask for xs
ks =  fftfreq(m, d=dx)

_f = fftshift(_f) # rearrange both arrays, because they are given in the conventional order
ks = fftshift(ks)

A = np.real(_f)
B = np.imag(_f)

In [ ]:
# reconstruction without doing an explicit Fourier synthesis

# f = irfft(_f, m) # or:
# f = ifft(ifftshift(_f), m) # note we need to shift it back to use ifft